In [2]:
%pip install statsmodels

  Using cached statsmodels-0.14.6-cp312-cp312-win_amd64.whl.metadata (9.8 kB)
  Using cached patsy-1.0.2-py2.py3-none-any.whl.metadata (3.6 kB)
Using cached statsmodels-0.14.6-cp312-cp312-win_amd64.whl (9.5 MB)
Using cached patsy-1.0.2-py2.py3-none-any.whl (233 kB)

   ---------------------------------------- 0/2 [patsy]
   -------------------- ------------------- 1/2 [statsmodels]
   -------------------- ------------------- 1/2 [statsmodels]
   -------------------- ------------------- 1/2 [statsmodels]
   -------------------- ------------------- 1/2 [statsmodels]
   -------------------- ------------------- 1/2 [statsmodels]
   -------------------- ------------------- 1/2 [statsmodels]
   -------------------- ------------------- 1/2 [statsmodels]
   -------------------- ------------------- 1/2 [statsmodels]
   -------------------- ------------------- 1/2 [statsmodels]
   -------------------- ------------------- 1/2 [statsmodels]
   -------------------- ------------------- 1/2 [statsmod

In [3]:

import pandas as pd
import numpy as np
from pathlib import Path
from statsmodels.stats.power import TTestIndPower

In [4]:
PILOT_CSV = r"C:\Users\Stemadmin\Desktop\Anuj Khadka\carbon_footprint\results\pilot_20260424_001142.csv"
RESULTS_DIR = Path(r"C:\Users\Stemadmin\Desktop\Anuj Khadka\carbon_footprint\results")

In [5]:
ENERGY_COL = "joules"
GROUP_COLS = ["language", "algorithm", "size"]
 

In [6]:
N_LANGUAGES       = 6
N_COMPARISONS     = (N_LANGUAGES * (N_LANGUAGES - 1)) // 2   # = 15
ALPHA_NOMINAL     = 0.05
ALPHA_CORRECTED   = ALPHA_NOMINAL / N_COMPARISONS             # Bonferroni
EFFECT_SIZE_D     = 0.5                                        # medium (Cohen 1988)
POWER_TARGET      = 0.80                                       # 80% power

In [7]:
# 1. Load pilot data

df = pd.read_csv(PILOT_CSV)
print(f"Loaded {len(df)} rows from pilot CSV.")
print(f"Columns: {list(df.columns)}\n")

Loaded 5400 rows from pilot CSV.
Columns: ['language', 'algorithm', 'size', 'run', 'joules', 'checksum']



In [8]:
# 2. Compute per-cell statistics
# --------------------------------------------------
stats = df.groupby(GROUP_COLS)[ENERGY_COL].agg(
    count="count",
    mean="mean",
    std="std"
).reset_index()
stats["cv_pct"] = (stats["std"] / stats["mean"]) * 100

In [9]:
# 3. Power analysis
    # --------------------------------------------------
    # TTestIndPower solves for sample size per group given:
    #   effect_size : Cohen's d
    #   alpha       : Bonferroni-corrected significance level
    #   power       : desired statistical power
    #   alternative : 'two-sided'
analysis = TTestIndPower()
n_required = analysis.solve_power(
    effect_size=EFFECT_SIZE_D,
    alpha=ALPHA_CORRECTED,
    power=POWER_TARGET,
    alternative="two-sided"
)
n_required_ceil = int(np.ceil(n_required))
 

In [10]:
separator = "=" * 65

In [11]:
print(separator)
print("POWER ANALYSIS PARAMETERS")
print(separator)
print(f"  Number of languages          : {N_LANGUAGES}")
print(f"  Pairwise comparisons (m)     : {N_COMPARISONS}")
print(f"  Nominal alpha                : {ALPHA_NOMINAL}")
print(f"  Bonferroni-corrected alpha'  : {ALPHA_CORRECTED:.6f}  (= {ALPHA_NOMINAL}/{N_COMPARISONS})")
print(f"  Target effect size (Cohen's d): {EFFECT_SIZE_D}  (medium)")
print(f"  Target power (1 - beta)      : {POWER_TARGET}")
print(f"  Test type                    : Two-sided Welch's t-test")
 

POWER ANALYSIS PARAMETERS
  Number of languages          : 6
  Pairwise comparisons (m)     : 15
  Nominal alpha                : 0.05
  Bonferroni-corrected alpha'  : 0.003333  (= 0.05/15)
  Target effect size (Cohen's d): 0.5  (medium)
  Target power (1 - beta)      : 0.8
  Test type                    : Two-sided Welch's t-test


In [12]:
print(f"\n{separator}")
print("RESULT")
print(separator)
print(f"  Raw N from solver            : {n_required:.4f}")
print(f"  Recommended N per cell       : {n_required_ceil}  (ceiling)")


RESULT
  Raw N from solver            : 116.2801
  Recommended N per cell       : 117  (ceiling)


In [13]:
print(f"\n{separator}")
print("PILOT STUDY VARIANCE SUMMARY")
print(separator)
print(f"  CV range   : {stats['cv_pct'].min():.2f}% -- {stats['cv_pct'].max():.2f}%")
print(f"  Mean CV    : {stats['cv_pct'].mean():.2f}%")
print(f"  Median CV  : {stats['cv_pct'].median():.2f}%")


PILOT STUDY VARIANCE SUMMARY
  CV range   : 4.40% -- 122.66%
  Mean CV    : 38.38%
  Median CV  : 43.03%


In [14]:
print(f"\n  Per-language mean CV:")
lang_cv = stats.groupby("language")["cv_pct"].mean().sort_values()
for lang, cv in lang_cv.items():
    print(f"    {lang:<12} : {cv:.2f}%")


  Per-language mean CV:
    python       : 28.85%
    javascript   : 36.40%
    java         : 38.48%
    go           : 40.27%
    c            : 42.31%
    rust         : 43.97%


In [16]:
# 5. Save results to file
# --------------------------------------------------
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
out_path = RESULTS_DIR / "power_analysis_results.txt"
with open(out_path, "w") as f:
    f.write("POWER ANALYSIS RESULTS\n")
    f.write(separator + "\n\n")
    f.write(f"Pilot CSV            : {PILOT_CSV}\n")
    f.write(f"N languages          : {N_LANGUAGES}\n")
    f.write(f"N comparisons (m)    : {N_COMPARISONS}\n")
    f.write(f"Alpha nominal        : {ALPHA_NOMINAL}\n")
    f.write(f"Alpha corrected      : {ALPHA_CORRECTED:.6f}\n")
    f.write(f"Effect size (d)      : {EFFECT_SIZE_D}\n")
    f.write(f"Power target         : {POWER_TARGET}\n")
    f.write(f"Raw N from solver    : {n_required:.4f}\n")
    f.write(f"Recommended N        : {n_required_ceil}\n\n")
    f.write("PAPER-READY SUMMARY\n")
    f.write(separator + "\n")

print(f"\nResults saved to: {out_path}")


Results saved to: C:\Users\Stemadmin\Desktop\Anuj Khadka\carbon_footprint\results\power_analysis_results.txt
